# 03 - U.S. candidate cohort preprocessing and official split

**Important design choice:** Instead of a new random split, intersect the *existing official global split* with the U.S. candidate IDs. This preserves your already untouched global test boundary, avoids accidental cross-experiment overlap, and will allow comparing global and U.S.-trained models on the same U.S. test cases later.

The new U.S. artifacts go only to `data/Processed-Data/us_only/`. The base one-hot data is saved for compatibility, but final `05/06` models **fit all transformations within cross-validation** using `utils.preprocessing.make_pipeline`.

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
GLOBAL_DIR = PROJECT_ROOT / "data" / "Processed-Data"
US_DIR = GLOBAL_DIR / "us_only"

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from utils.us_scope import align_to_global_split, us_state_proxy_mask
clean_path = US_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
for path in [clean_path, GLOBAL_DIR / "train_project_ids.csv", GLOBAL_DIR / "test_project_ids.csv"]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; run U.S. notebook 02 and global notebook 03 first")
clean_df = pd.read_csv(clean_path)
original_train = pd.read_csv(GLOBAL_DIR / "train_project_ids.csv")
original_test = pd.read_csv(GLOBAL_DIR / "test_project_ids.csv")
assert clean_df["project id"].is_unique and us_state_proxy_mask(clean_df).all()
train_ids, test_ids = align_to_global_split(clean_df, original_train, original_test)
print("Provisional U.S. training:", len(train_ids), "test:", len(test_ids))
print("Training success rate:", train_ids["target"].mean())

Provisional U.S. training: 26945 test: 11475
Training success rate: 0.546149563926517


In [2]:
by_id = clean_df.set_index("project id", drop=False)
train_rows = by_id.loc[train_ids["project id"]].copy().reset_index(drop=True)
test_rows = by_id.loc[test_ids["project id"]].copy().reset_index(drop=True)
assert set(train_rows["project id"]).isdisjoint(test_rows["project id"])
assert len(train_rows) + len(test_rows) == clean_df["status"].isin(["successful", "failed"]).sum()
assert train_rows["status"].eq("successful").astype(int).tolist() == train_ids["target"].tolist()
assert test_rows["status"].eq("successful").astype(int).tolist() == test_ids["target"].tolist()
X_train = train_rows.drop(columns=["status"])
X_test = test_rows.drop(columns=["status"])
y_train, y_test = train_ids["target"], test_ids["target"]
print("Split membership and target agreement checked; no new split created.")

Split membership and target agreement checked; no new split created.


In [3]:
# Base preprocessing is for documentation only. 05 refits its own model pipeline per fold.
NUMERIC = ["goal", "levels", "duration"]
CATEGORICAL = ["category", "subcategory"]
base_preprocessor = ColumnTransformer([
    ("numerical", Pipeline([("impute", SimpleImputer(strategy="median")),
                            ("scale", StandardScaler())]), NUMERIC),
    ("categorical", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                              ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL),
], verbose_feature_names_out=True)
base_train_array = base_preprocessor.fit_transform(X_train)
base_test_array = base_preprocessor.transform(X_test)
cols = base_preprocessor.get_feature_names_out()
train_base = pd.DataFrame(base_train_array, columns=cols)
test_base = pd.DataFrame(base_test_array, columns=cols)
assert train_base.shape[1] == test_base.shape[1]
print("Base processed train/test shapes:", train_base.shape, test_base.shape)

Base processed train/test shapes: (26945, 65) (11475, 65)


In [4]:
US_DIR.mkdir(parents=True, exist_ok=True)
train_base.to_csv(US_DIR / "X_train_base_preprocessed.csv", index=False)
test_base.to_csv(US_DIR / "X_test_base_preprocessed.csv", index=False)
train_ids.to_csv(US_DIR / "train_project_ids.csv", index=False)
test_ids.to_csv(US_DIR / "test_project_ids.csv", index=False)
pd.DataFrame({"target": y_train}).to_csv(US_DIR / "y_train.csv", index=False)
pd.DataFrame({"target": y_test}).to_csv(US_DIR / "y_test.csv", index=False)
joblib.dump(base_preprocessor, US_DIR / "base_preprocessor.joblib")
print("Saved 7 U.S. subset split/preprocessing artifacts in", US_DIR)
print("Note: official U.S. test IDs and labels are now persisted, but must not be used in 04/05.")

Saved 7 U.S. subset split/preprocessing artifacts in /Users/annasehgal/State-Street-1C-kickstarter-crowdfunding-recommendation-engine/data/Processed-Data/us_only
Note: official U.S. test IDs and labels are now persisted, but must not be used in 04/05.


**Handoff:** Next, `04_feature_engineering.ipynb` analyzes *only U.S. training campaigns*. The U.S. scope remains provisional until advisor/source approval. This notebook intentionally never evaluates the test outcomes.